# 02c — Train Swin Tiny (PyTorch)

Binary coffee-bean classification (`defect` vs `non_defect`) on the Robusta
Lampung dataset cropped by `01_auto_crop.py`. This notebook trains **only**
the `swin_t` backbone (Swin Transformer Tiny — shifted-window self-attention
SOTA) using the pre-computed deterministic split from `02_prepare_split.ipynb`.

**Prerequisites**: Run `02_prepare_split.ipynb` first to generate
`reports/split_indices.json`.

**Run this on Google Colab with a GPU runtime** (Runtime → Change runtime
type → T4 GPU). Training will fall back to CPU if no GPU is available,
but expect ~30× slower epochs.

## 1 · Environment setup

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    # Upgrade torch, torchvision, and torchaudio together to maintain consistency
    # and fix the Python 3.12 OverflowError in ColorJitter
    !pip -q install --upgrade torch==2.4.* torchvision==0.19.* torchaudio==2.4.* \
        scikit-learn==1.4.* seaborn==0.13.* matplotlib==3.8.*

Running in Colab: True


In [2]:
# ---- Mount Google Drive (Colab only) -------------------------------------
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
    WORK_ROOT  = Path('/content/work')   # fast local SSD on Colab
else:
    # Local fallback (e.g., when running this notebook on the user's PC).
    DRIVE_ROOT = Path('..').resolve()
    WORK_ROOT  = DRIVE_ROOT

WORK_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)
print('WORK_ROOT  =', WORK_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT = /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah
WORK_ROOT  = /content/work


In [ ]:
import zipfile

DATA_ZIP  = DRIVE_ROOT / 'Dataset_Cropped.zip'
DATA_ROOT = WORK_ROOT

# Check if a representative subfolder of the dataset exists
if not (DATA_ROOT / 'defect').exists():
    if IN_COLAB:
        if not DATA_ZIP.exists():
            raise FileNotFoundError(
                f'Dataset zip not found.\n'
                f'Please upload Dataset_Cropped.zip to:\n  {DATA_ZIP}'
            )
        print(f'Extracting {DATA_ZIP} -> {WORK_ROOT} ...')
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(WORK_ROOT)
        print('Done.')
    else:
        raise FileNotFoundError(
            f'Dataset folder not found at:\n  {DATA_ROOT}\n'
            f'Please run 01_auto_crop.py first to generate Dataset_Cropped/.'
        )
else:
    print(f'Dataset already present at {DATA_ROOT} — skipping extract.')

# Sanity check class folders
for c in ('defect', 'non_defect'):
    folder = DATA_ROOT / c
    if not folder.exists():
        raise FileNotFoundError(
            f'Expected class folder not found: {folder}\n'
            f'Dataset_Cropped/ must contain defect/ and non_defect/ subfolders.'
        )
    n = len(list(folder.glob('*.jpg')))
    print(f'  {c:>10}: {n:>5} images')

Dataset already present at /content/work/Dataset_Cropped — skipping extract.
      defect:   500 images
  non_defect:   500 images


In [4]:
import importlib, shutil, sys, os

src_utils = DRIVE_ROOT / 'scripts' / 'utils.py'
if src_utils.exists():
    shutil.copy(src_utils, '/content/utils.py' if IN_COLAB else 'utils.py')
    sys.path.insert(0, '/content' if IN_COLAB else '.')
else:
    print(f'[warn] {src_utils} not found — paste utils.py manually.')

# Fix the OverflowError by modifying utils.py directly if it exists
utils_path = '/content/utils.py' if IN_COLAB else 'utils.py'
if os.path.exists(utils_path):
    with open(utils_path, 'r') as f:
        content = f.read()
    # Set hue to 0 in ColorJitter to avoid the Python 3.12/Pillow bug
    fixed_content = content.replace("hue=0.1", "hue=0.0")
    with open(utils_path, 'w') as f:
        f.write(fixed_content)

import utils  # noqa: E402
importlib.reload(utils)
from utils import (
    NUM_CLASSES, IMG_SIZE,
    seed_everything, get_transforms, TransformSubset,
    build_model, count_trainable_params, count_total_params,
)

## 2 · Reproducibility & hyperparameters

In [5]:
import torch

SEED = 42
seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'           # mixed precision only on GPU
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # auto-tune convolutions for fixed 224x224 input
print('Device:', DEVICE, '| AMP:', USE_AMP)
if DEVICE.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))

Device: cuda | AMP: True
GPU   : Tesla T4


In [6]:
HPARAMS = {
    'batch_size'   : 48,   # T4 has 16 GB VRAM — Swin-T uses more memory due to attention; 48 is safe
    'num_workers'  : 4,    # Colab has multi-core CPU; 4 workers saturates the GPU
    'epochs'       : 25,
    'learning_rate': 1e-4,
    'weight_decay' : 1e-4,
    'patience'     : 5,
    'split_ratios' : (0.8, 0.1, 0.1),
    'seed'         : 42,
    'img_size'     : 224,
}
HPARAMS

{'batch_size': 48,
 'num_workers': 4,
 'epochs': 25,
 'learning_rate': 0.0001,
 'weight_decay': 0.0001,
 'patience': 5,
 'split_ratios': (0.8, 0.1, 0.1),
 'seed': 42,
 'img_size': 224}

## 3 · Load pre-computed split & build DataLoaders

The deterministic 80/10/10 split was generated by `02_prepare_split.ipynb`
and saved to `reports/split_indices.json`. We load the indices and
reconstruct subsets via `Subset` (no `random_split` here).

In [7]:
import json
from torch.utils.data import DataLoader, Subset
from torchvision import datasets

# Load the full dataset (transform=None — we apply per-subset transforms later)
full_dataset = datasets.ImageFolder(root=str(DATA_ROOT), transform=None)
print('Classes (folder-alphabetic):', full_dataset.class_to_idx)
print('Total images:', len(full_dataset))

Classes (folder-alphabetic): {'defect': 0, 'non_defect': 1}
Total images: 1000


In [8]:
# ---- Load and validate split_indices.json --------------------------------
split_path = DRIVE_ROOT / 'reports' / 'split_indices.json'
if not split_path.exists():
    raise FileNotFoundError("Run 02_prepare_split.ipynb first")

with open(split_path) as f:
    split = json.load(f)

# Validate basenames match current dataset
saved_basenames = [Path(p).name for p in split['samples']]
current_samples = [s for s, _ in full_dataset.samples]
current_basenames = [Path(p).name for p in current_samples]

if saved_basenames != current_basenames:
    raise ValueError(
        'Dataset basenames do not match split_indices.json. '
        'The dataset may have changed. Re-run 02_prepare_split.ipynb.'
    )

print(f"Split loaded: train={len(split['train_indices'])} | "
      f"val={len(split['val_indices'])} | test={len(split['test_indices'])}")

Split loaded: train=800 | val=100 | test=100


In [9]:
# Reconstruct subsets from saved indices (NOT random_split)
train_subset = Subset(full_dataset, split['train_indices'])
val_subset   = Subset(full_dataset, split['val_indices'])

# Wrap each subset with the appropriate transform
train_ds = TransformSubset(train_subset, get_transforms(train=True))
val_ds   = TransformSubset(val_subset,   get_transforms(train=False))

common = dict(batch_size=HPARAMS['batch_size'],
              num_workers=HPARAMS['num_workers'],
              pin_memory=DEVICE.type == 'cuda')

train_loader = DataLoader(train_ds, shuffle=True,  drop_last=False, **common)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **common)

print('train batches:', len(train_loader),
      '| val batches:', len(val_loader))

train batches: 17 | val batches: 3


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


## 4 · Training loop

Single-model training with:
- **AdamW** optimizer
- **CrossEntropyLoss** (binary task framed as 2-class softmax)
- **CosineAnnealingLR** scheduler (T_max = epochs)
- **Mixed-precision** (AMP) on GPU
- **Early stopping** on validation accuracy with `patience` epochs
- Per-epoch progress: elapsed time, ETA, `*` for improvements

In [10]:
import copy, time
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

MODEL_NAME = "swin_t"


def run_epoch(model, loader, criterion, optimizer, scaler, train: bool):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(x)
                loss   = criterion(logits, y)
            if train:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += x.size(0)
    return loss_sum / total, correct / total

In [11]:
import os, sys, importlib, copy, time
import numpy as np
from PIL import Image
import torchvision.transforms.functional as F

# --- PERMANENT FIX FOR PYTHON 3.12 OVERFLOW ERROR ---
# We monkey-patch the internal torchvision PIL-based hue adjustment
try:
    import torchvision.transforms._functional_pil as F_pil
    def safe_adjust_hue(img, hue_factor):
        if not (0.5 >= hue_factor >= -0.5):
            raise ValueError(f"hue_factor ({hue_factor}) is not in [-0.5, 0.5].")
        if not F._is_pil_image(img):
            return F.adjust_hue(img, hue_factor)

        input_mode = img.mode
        if input_mode in ("L", "RGB", "RGBA"):
            res = img.convert("HSV")
            np_h, s, v = res.split()
            np_h = np.array(np_h, dtype=np.int16) # Use int16 to prevent overflow
            shift = int(hue_factor * 255)
            np_h = (np_h + shift) % 255 # Wrap around safely
            np_h = Image.fromarray(np_h.astype(np.uint8), "L")
            res = Image.merge("HSV", (np_h, s, v)).convert(input_mode)
            return res
        return F.adjust_hue(img, hue_factor)

    F_pil.adjust_hue = safe_adjust_hue
    print("Successfully patched torchvision.adjust_hue for stability.")
except Exception as e:
    print(f"Patching failed: {e}. Moving on...")
# -----------------------------------------------------

print(f'\n========== {MODEL_NAME} ==========')
seed_everything(HPARAMS['seed'])

# Force reload
from torch.utils.data import DataLoader
import utils
importlib.reload(utils)
from utils import get_transforms, TransformSubset

train_ds = TransformSubset(train_subset, get_transforms(train=True))
val_ds   = TransformSubset(val_subset,   get_transforms(train=False))

common = dict(batch_size=HPARAMS['batch_size'],
              num_workers=0,
              pin_memory=DEVICE.type == 'cuda')

train_loader = DataLoader(train_ds, shuffle=True,  drop_last=False, **common)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **common)

model     = build_model(MODEL_NAME).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(),
                  lr=HPARAMS['learning_rate'],
                  weight_decay=HPARAMS['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=HPARAMS['epochs'])
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print(f"Params: total={count_total_params(model):,} | trainable={count_trainable_params(model):,}")

history = []; best_val_acc, best_epoch = -1.0, -1; best_state = None; epochs_no_improve = 0

t0 = time.time()
for epoch in range(1, HPARAMS['epochs'] + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scaler, train=True)
    vl_loss, vl_acc = run_epoch(model, val_loader, criterion, optimizer, scaler, train=False)
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_acc': tr_acc, 'val_loss': vl_loss, 'val_acc': vl_acc, 'lr': optimizer.param_groups[0]['lr']})

    improved = vl_acc > best_val_acc
    flag = '*' if improved else ' '
    elapsed = time.time() - t0
    elapsed_str = time.strftime('%H:%M:%S', time.gmtime(elapsed))

    print(f'  E{epoch:02d}{flag} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | elapsed={elapsed_str}')

    if improved:
        best_val_acc, best_epoch = vl_acc, epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= HPARAMS['patience']:
            print(f'  Early stop at epoch {epoch}.')
            break

print(f'  Training complete: best_val_acc={best_val_acc:.4f}')

Successfully patched torchvision.adjust_hue for stability.

========== swin_t ==========


/tmp/ipykernel_11396/3521543562.py:59: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
/tmp/ipykernel_11396/3521543562.py:23: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  np_h = Image.fromarray(np_h.astype(np.uint8), "L")


Params: total=27,520,892 | trainable=27,520,892


/tmp/ipykernel_11396/4238972682.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


  E01* | tr_loss=0.4780 tr_acc=0.7538 | vl_loss=0.2276 vl_acc=0.9300 | elapsed=00:00:13
  E02  | tr_loss=0.3294 tr_acc=0.8712 | vl_loss=0.4179 vl_acc=0.7800 | elapsed=00:00:26
  E03  | tr_loss=0.2283 tr_acc=0.9050 | vl_loss=0.3372 vl_acc=0.8400 | elapsed=00:00:37
  E04* | tr_loss=0.1945 tr_acc=0.9175 | vl_loss=0.1179 vl_acc=0.9700 | elapsed=00:00:48
  E05  | tr_loss=0.1298 tr_acc=0.9487 | vl_loss=0.1610 vl_acc=0.9200 | elapsed=00:01:00
  E06  | tr_loss=0.0930 tr_acc=0.9637 | vl_loss=0.0728 vl_acc=0.9700 | elapsed=00:01:12
  E07  | tr_loss=0.0771 tr_acc=0.9750 | vl_loss=0.1829 vl_acc=0.9300 | elapsed=00:01:24
  E08  | tr_loss=0.1685 tr_acc=0.9275 | vl_loss=0.0812 vl_acc=0.9700 | elapsed=00:01:35
  E09* | tr_loss=0.1015 tr_acc=0.9637 | vl_loss=0.0648 vl_acc=0.9800 | elapsed=00:01:48
  E10  | tr_loss=0.0918 tr_acc=0.9613 | vl_loss=0.0800 vl_acc=0.9800 | elapsed=00:01:59
  E11  | tr_loss=0.0741 tr_acc=0.9725 | vl_loss=0.0647 vl_acc=0.9700 | elapsed=00:02:10
  E12  | tr_loss=0.0544 tr_acc=0

## 5 · Save artefacts

In [13]:
# ---- Checkpoint ----------------------------------------------------------
ckpt_path = DRIVE_ROOT / 'checkpoints' / f'best_model_{MODEL_NAME}.pth'
torch.save({
    'model_name'  : MODEL_NAME,
    'state_dict'  : best_state,
    'best_epoch'  : best_epoch,
    'best_val_acc': best_val_acc,
    'hparams'     : HPARAMS,
    'class_to_idx': full_dataset.class_to_idx,
}, ckpt_path)
print(f'Saved checkpoint -> {ckpt_path}  '
      f'(epoch {best_epoch}, val_acc={best_val_acc:.4f})')

# ---- History CSV ----------------------------------------------------------
hist_df = pd.DataFrame(history)
hist_csv_path = DRIVE_ROOT / 'reports' / f'history_{MODEL_NAME}.csv'
hist_df.to_csv(hist_csv_path, index=False)
print(f'Saved history  -> {hist_csv_path}')

# ---- Timing JSON ----------------------------------------------------------
import json as _json
timing_path = DRIVE_ROOT / 'reports' / f'timing_{MODEL_NAME}.json'

# Fix: use 'elapsed' which exists in the kernel instead of the missing 'train_secs'
timing_path.write_text(_json.dumps({
    'model_name'  : MODEL_NAME,
    'train_secs'  : round(elapsed, 2),
    'best_epoch'  : best_epoch,
    'best_val_acc': round(best_val_acc, 4),
}, indent=2))
print(f'Saved timing   -> {timing_path}')

Saved checkpoint -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/checkpoints/best_model_swin_t.pth  (epoch 9, val_acc=0.9800)
Saved history  -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/history_swin_t.csv
Saved timing   -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/timing_swin_t.json


In [14]:
# ---- Per-model curves PNG ------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='val')
axes[0].set(title=f'{MODEL_NAME} — Loss', xlabel='epoch', ylabel='loss')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(hist_df['epoch'], hist_df['train_acc'], label='train')
axes[1].plot(hist_df['epoch'], hist_df['val_acc'],   label='val')
axes[1].set(title=f'{MODEL_NAME} — Accuracy', xlabel='epoch', ylabel='accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()
curves_path = DRIVE_ROOT / 'reports' / f'curves_{MODEL_NAME}.png'
fig.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved curves   -> {curves_path}')

Saved curves   -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/curves_swin_t.png


---

✅ **Swin Tiny training complete.** Artefacts saved:
- `checkpoints/best_model_swin_t.pth`
- `reports/history_swin_t.csv`
- `reports/curves_swin_t.png`
- `reports/timing_swin_t.json`

Run `02d_compare_training.ipynb` after all three models are trained to
generate the cross-model comparison artefacts.